In [1]:
pip install sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
from sqlalchemy import create_engine,text
import psycopg2

In [4]:
username = "postgres"     
password = "admin"     
host = "localhost"            
port = "5434"                   
database = "db_northwind"     


# Create the SQLAlchemy engine
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")


## Get all order date in a list

In [7]:
query = text("SELECT DISTINCT orderdate FROM public.fct_tblnorthwind;")
# Execute and fetch results
with engine.connect() as conn:
    result = conn.execute(query)
    dates = [row[0] for row in result.fetchall()]  # Extract first column values

dates_str = [d.strftime("%Y-%m-%d") for d in dates]

print(dates_str)

['1997-04-07', '1996-11-15', '1996-09-09', '1997-04-25', '1996-09-05', '1997-05-21', '1997-05-09', '1997-05-28', '1997-03-14', '1997-04-08', '1996-09-27', '1996-08-21', '1997-03-12', '1996-08-07', '1996-12-18', '1996-08-14', '1996-08-05', '1997-06-06', '1996-08-12', '1996-08-16', '1997-03-31', '1996-08-29', '1996-10-29', '1997-06-24', '1997-01-30', '1996-12-09', '1997-03-28', '1997-03-04', '1997-06-03', '1997-07-28', '1997-02-21', '1997-01-13', '1997-03-11', '1997-05-29', '1996-11-29', '1997-07-02', '1996-11-28', '1997-06-23', '1997-04-21', '1997-01-29', '1997-03-25', '1996-07-09', '1996-10-07', '1997-07-03', '1997-01-03', '1996-11-11', '1997-04-22', '1997-07-01', '1996-07-23', '1997-03-10', '1996-11-01', '1996-12-23', '1997-07-31', '1996-08-22', '1996-09-13', '1997-03-24', '1996-12-20', '1997-08-07', '1997-03-06', '1997-02-12', '1997-08-04', '1997-02-25', '1997-07-17', '1996-11-22', '1997-02-04', '1997-06-25', '1997-04-23', '1996-11-08', '1997-01-01', '1996-08-02', '1997-01-24', '1996

## Make an api call for waether information. handle all cases

In [9]:
import requests
import pandas as pd
import time

# order dates from the database
dates = dates_str

# Location coordinates for Berlin
latitude = 52.5200
longitude = 13.4050

# Function to fetch avg temp for a given date
def get_avg_temp(date, lat, lon, retries=3, backoff=2):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={date}&end_date={date}"
        "&hourly=temperature_2m"
    )
    
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=10)  # set timeout (10s)
            response.raise_for_status()              # raise HTTPError if bad status
            data = response.json()
            
            if "hourly" in data and "temperature_2m" in data["hourly"]:
                temps = data["hourly"]["temperature_2m"]
                avg_temp = sum(temps) / len(temps)
                return round(avg_temp, 2)
            else:
                return None
        
        except requests.exceptions.ReadTimeout:
            print(f"Timeout for date {date}, retrying... ({attempt+1}/{retries})")
            time.sleep(backoff * (attempt + 1))  # exponential backoff
            
        except requests.exceptions.RequestException as e:
            print(f"Error for date {date}: {e}")
            return None
    
    return None  # if all retries fail

# Loop through dates
results = []
for d in dates:
    avg_temp = get_avg_temp(d, latitude, longitude)
    results.append({"date": d, "avg_temp": avg_temp})

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)


Timeout for date 1997-04-14, retrying... (1/3)
Timeout for date 1996-12-03, retrying... (1/3)
           date  avg_temp
0    1997-04-07      2.14
1    1996-11-15      3.68
2    1996-09-09     11.25
3    1997-04-25      9.24
4    1996-09-05     13.53
..          ...       ...
282  1996-09-12     10.90
283  1997-02-13      7.10
284  1997-05-02     13.93
285  1996-10-11      8.35
286  1996-07-11     14.50

[287 rows x 2 columns]


In [10]:
df.to_sql(
    name="weather_data",   # table name in postgres
    con=engine,
    if_exists="replace",   # options: 'fail', 'replace', 'append'
    index=False            # do not write pandas index as a column
)

287

## Extract quotes and store in a table.

In [17]:
import requests
from bs4 import BeautifulSoup
import random
import pandas as pd
# 1. Scrape quotes from "quotes.toscrape.com"
def scrape_quotes():
    quotes = []
    base_url = "http://quotes.toscrape.com/page/{}/"
    
    for page in range(1, 6):  # scrape first 5 pages (~50 quotes)
        url = base_url.format(page)
        resp = requests.get(url, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
        
        for quote in soup.find_all("span", class_="text"):
            quotes.append(quote.get_text(strip=True))
    
    return quotes



quotes_list = scrape_quotes()

In [18]:
df_quotes = pd.DataFrame(quotes_list, columns=["quote"])
df_quotes.to_sql(
    name="quotes_table",   # table name in postgres
    con=engine,
    if_exists="replace",   # replace if table already exists
    index=False
)

50

## Above code completes all the data enrichment needed for exercise. In below part using enriched data create your tables.

## Add weather information in final fact table

In [19]:
alter_col = """ALTER TABLE public.fct_tblnorthwind
ADD COLUMN avg_temp_order_date NUMERIC;"""

In [20]:
with engine.connect() as conn:
    conn.execute(text(alter_col))
    conn.commit()

## Merge weather information with final fact model

In [25]:
merge_sql = """MERGE INTO public.fct_tblnorthwind AS f
USING public.weather_data AS w
ON f.orderdate = cast(w.date as date)
WHEN MATCHED THEN
  UPDATE SET avg_temp_order_date = w.avg_temp ;"""

In [26]:
with engine.connect() as conn:
    conn.execute(text(merge_sql))
    conn.commit()

## Add a new column for quotes

In [27]:
rand_quote_sql = """
ALTER TABLE public.fct_tblnorthwind
ADD COLUMN random_quote TEXT;"""

In [28]:
with engine.connect() as conn:
    conn.execute(text(rand_quote_sql))
    conn.commit()

## Add a random quote for each execution

In [33]:
add_quote = """UPDATE public.fct_tblnorthwind f
SET random_quote = (
    SELECT quote 
    FROM public.quotes_table 
    ORDER BY RANDOM() 
    LIMIT 1
);"""

In [34]:
with engine.connect() as conn:
    conn.execute(text(add_quote))
    conn.commit()

## In below part we will create reporting tables with enriched data 

In [ ]:
sql_cust = """CREATE TABLE IF NOT EXISTS customers AS
SELECT DISTINCT customerid, companyname, contactname, contacttitle,random_quote
FROM fct_tblnorthwind;"""

In [40]:
with engine.connect() as conn:
    conn.execute(text(sql_cust))
    conn.commit()

In [41]:
sql_order = """CREATE TABLE IF NOT EXISTS orders AS
SELECT DISTINCT orderid,
      customerid,
     orderdate,
requireddate,
     shippeddate,
quantity,
avg_temp_order_date
FROM fct_tblnorthwind"""

In [42]:
with engine.connect() as conn:
    conn.execute(text(sql_order))
    conn.commit()

## Load tables to Metabase.
https://www.metabase.com/

Run `docker run -d -p 3000:3000 metabase/metabase` to pull latest metabase image and run it through Docker

Access Metabase from http://localhost:3000/

Provide your details and connect to postgres db from metabase.

Navigate to Metabase --> Databases. Check all the tables successfully loaded to Metabase.